# Complexité ajouté

In [3]:
import sys, os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from src.evaluation.evaluator import TimeSeriesEvaluator
from src.models.prophet import ProphetRegressor
from src.models.fourier import FourierRegressor
from src.models.baselines import AverageRegressor, LastWeekRegressor, YesterdayRegressor, LinearRegressor, SimilarDayRegressor
from src.models.xgboost_regressor import XGBoostRegressor
from src.models.lstm_regressor import LSTMRegressor
from src.models.sarimax import SARIMAXRegressor

df = pd.read_csv('../data/processed/dataset_final.csv')

evaluator = TimeSeriesEvaluator(df, "realistic")

val_start = pd.to_datetime('2024-01-01').tz_localize('Europe/Paris')
test_start = pd.to_datetime('2025-01-01').tz_localize('Europe/Paris')

df_train, df_val, df_test = evaluator.split_data(val_start, test_start, load_existing_splits=True)

CONSUMPTION_COLS = [
    'consommation_mw_tendance_24h_derniere_connue',
    'consommation_mw_tendance_6h_derniere_connue',
    'consommation_mw_min_24h_derniere_connue', 'consommation_mw_max_24h_derniere_connue',
    'consommation_mw_moyenne_24h_derniere_connue',
    'consommation_mw_meme_heure_derniere_connue',
    'consommation_mw_derniere_connue',
    'consommation_mw_moins_7', 'consommation_mw_moins_365', 'consommation_mw_moins_366',
]

TIME_COLS = [
    'cible_heure_sin', 'cible_heure_cos',
    'cible_jour_semaine_sin','cible_jour_semaine_cos',
    'cible_jour_annee_sin','cible_jour_annee_cos',
]

FERIE_COLS = [
    'heure', 'jour_semaine', 'jour_annee', 'mois', 'saison','saison_meteorologique', 'annee',
    'weekend', 'ferie', 'vacances', 'confinement_numero',
]

TEMPERATURE_COLS = [
	'temperature_c_pondere_pop_meme_heure_derniere_connue',
	'temperature_c_pondere_pop_derniere_connue',
	'temperature_c_pondere_pop_moins_365', 'temperature_c_pondere_pop_moins_366',
	'temperature_c_pondere_pop_moyenne_24h_derniere_connue',
	'temperature_c_pondere_pop_min_24h_derniere_connue',
	'temperature_c_pondere_pop_max_24h_derniere_connue',
	'temperature_c_pondere_pop_moyenne_48h_derniere_connue',
	'temperature_c_pondere_pop_moyenne_72h_derniere_connue',
	'temperature_c_pondere_pop_tendance_6h_derniere_connue',
	'temperature_c_pondere_pop_tendance_24h_derniere_connue',
	'temperature_c_pondere_pop_sous_15_derniere_connue',
	'temperature_c_pondere_pop_au_dessus_22_derniere_connue',
	'temperature_c_pondere_pop_sous_15_cumul_3j_derniere_connue',
	'temperature_c_pondere_pop_ecart_normale_derniere_connue'
]

In [4]:
ALL_COLS = TIME_COLS + CONSUMPTION_COLS + FERIE_COLS + TEMPERATURE_COLS
NO_CONS = TIME_COLS + FERIE_COLS + TEMPERATURE_COLS
NO_CAL = TIME_COLS + CONSUMPTION_COLS + TEMPERATURE_COLS
NO_TEMP = TIME_COLS + CONSUMPTION_COLS + FERIE_COLS

groups = {
    "All Features": ALL_COLS,
    "No Consumption": NO_CONS,
    "No Calendar": NO_CAL,
    "No Temperature": NO_TEMP
}

results = []

print("--- XGBoost Ablation ---")
for group_name, cols in groups.items():
    print(f"\n>>> Testing XGBoost: {group_name}")
    xgb_grid = [{"colonnes_features": [cols]}]
    res_list, _ = evaluator.grid_search_rolling_validation(
        XGBoostRegressor,
        xgb_grid,
        metric="MAE",
        eval_train=False
    )
    mae = res_list[0]['score']
    results.append({"Model": "XGBoost", "Features": group_name, "MAE (MW)": mae})

# --- 2. Fourier Ablation ---
print("\n--- Fourier Ablation ---")
for group_name, cols in groups.items():
    print(f"\n>>> Testing Fourier: {group_name}")
    fourier_grid = [{"features_cols": [cols]}]
    res_list, _ = evaluator.grid_search_rolling_validation(
        FourierRegressor,
        fourier_grid,
        metric="MAE",
        eval_train=False
    )
    mae = res_list[0]['score']
    results.append({"Model": "Fourier", "Features": group_name, "MAE (MW)": mae})

# --- Summary Table ---
df_results = pd.DataFrame(results)

base_maes = df_results[df_results['Features'] == 'All Features'].set_index('Model')['MAE (MW)']
df_results['MAE_Base'] = df_results['Model'].map(base_maes)
df_results['Increase vs Base (MW)'] = df_results['MAE (MW)'] - df_results['MAE_Base']
df_results['Error Rise (%)'] = (df_results['Increase vs Base (MW)'] / df_results['MAE_Base'] * 100).round(2)

df_results = df_results.drop(columns=['MAE_Base'])

display(df_results.sort_values(['Model', 'Increase vs Base (MW)']))


--- XGBoost Ablation ---

>>> Testing XGBoost: All Features
Grid Search de XGBoostRegressorCustom :

Combination 1/1 : {'colonnes_features': ['cible_heure_sin', 'cible_heure_cos', 'cible_jour_semaine_sin', 'cible_jour_semaine_cos', 'cible_jour_annee_sin', 'cible_jour_annee_cos', 'consommation_mw_tendance_24h_derniere_connue', 'consommation_mw_tendance_6h_derniere_connue', 'consommation_mw_min_24h_derniere_connue', 'consommation_mw_max_24h_derniere_connue', 'consommation_mw_moyenne_24h_derniere_connue', 'consommation_mw_meme_heure_derniere_connue', 'consommation_mw_derniere_connue', 'consommation_mw_moins_7', 'consommation_mw_moins_365', 'consommation_mw_moins_366', 'heure', 'jour_semaine', 'jour_annee', 'mois', 'saison', 'saison_meteorologique', 'annee', 'weekend', 'ferie', 'vacances', 'confinement_numero', 'temperature_c_pondere_pop_meme_heure_derniere_connue', 'temperature_c_pondere_pop_derniere_connue', 'temperature_c_pondere_pop_moins_365', 'temperature_c_pondere_pop_moins_366', 't

,Model,Features,MAE (MW),Increase vs Base (MW),Error Rise (%)
4,Fourier,All Features,1602.059074,0.000000,0.00
7,Fourier,No Temperature,1767.602661,165.543587,10.33
6,Fourier,No Calendar,1771.114015,169.054941,10.55
5,Fourier,No Consumption,2150.207055,548.147981,34.22
0,XGBoost,All Features,1349.194997,0.000000,0.00
3,XGBoost,No Temperature,1371.523279,22.328282,1.65
2,XGBoost,No Calendar,1451.042577,101.847580,7.55
1,XGBoost,No Consumption,1481.705549,132.510552,9.82


On peut aussi analyser des importances des variables dans 02_evaluation.